In [1]:
# Instalar todas las librerías requeridas de un solo golpe
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib streamlit pyngrok

In [2]:
"""
CENTINELA AGRO - Paso 1: ETL (limpieza y union de las 4 fuentes)
Compatible con Google Colab y ejecución local en PC.
"""
import pandas as pd, numpy as np, os, sys

# Detección inteligente de entorno (Drive vs Carpeta Local)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    RAW = "/content/drive/MyDrive/datos_crudos/"
except Exception:
    RAW = "./datos_crudos/"

OUT = "datos_limpios/"; os.makedirs(OUT, exist_ok=True)
MESES = {'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,'jul':7,'ago':8,'sep':9,'oct':10,'nov':11,'dic':12}

# Verificar existencia de archivos crudos antes de procesar
for archivo in ["precios_sipsa_raw.csv", "clima_raw.csv", "trm_raw.csv", "produccion_raw.csv"]:
    ruta_completa = os.path.join(RAW, archivo)
    if not os.path.exists(ruta_completa):
        print(f"⚠️ Alerta: No se encontró el archivo '{archivo}' en la ruta: {RAW}")

# ---------------- 1. SIPSA (precios) ----------------
if os.path.exists(RAW+"precios_sipsa_raw.csv"):
    raw = pd.read_csv(RAW+"precios_sipsa_raw.csv", header=None, dtype=str, low_memory=False).iloc[1:]
    raw = raw[raw[0] != "Fecha"]
    es24 = raw[5].str.contains("24", na=False)
    a = raw[~es24][[0,1,2,3,4]]; a.columns = ["fecha","grupo","producto","mercado","precio"]
    b = raw[es24][[0,1,2,4,6]];  b.columns = ["fecha","grupo","producto","mercado","precio"]
    sipsa = pd.concat([a,b], ignore_index=True)

    def parse_fecha(s):
        s = str(s).strip()
        if "/" in s:
            d,m,y = s.split("/"); return pd.Timestamp(int(y), int(m), 1)
        m,y = s.split("-"); return pd.Timestamp(2000+int(y), MESES[m.lower()], 1)

    sipsa["mes"] = sipsa.fecha.map(parse_fecha)
    sipsa["precio"] = pd.to_numeric(sipsa.precio.str.strip().str.replace(".","",regex=False), errors="coerce")
    sipsa["producto"] = sipsa.producto.str.strip().str.lower()
    sipsa["grupo"] = sipsa.grupo.str.strip().str.upper()
    n_dup = sipsa.duplicated(["mes","producto","mercado"]).sum()
    sipsa = sipsa.groupby(["mes","grupo","producto","mercado"], as_index=False).precio.mean()
    print(f"SIPSA: {len(sipsa):,} filas limpias | repetidos resueltos: {n_dup} | productos: {sipsa.producto.nunique()}")
    cma = sipsa[sipsa.mercado.str.contains("Central Mayorista de Antioquia", na=False)].drop(columns="mercado")
    cma.to_csv(OUT+"precios_cma_limpio.csv", index=False)

# ---------------- 2. Clima ----------------
if os.path.exists(RAW+"clima_raw.csv"):
    cl = pd.read_csv(RAW+"clima_raw.csv", parse_dates=["time"])
    cl = cl[(cl.time >= "2020-01-01") & (cl.time <= "2024-12-31")]
    cl["mes"] = cl.time.dt.to_period("M").dt.to_timestamp()
    por_zona = cl.groupby(["mes","subregion"]).agg(lluvia_mm=("precipitation_sum","sum"),
                temp_media=("temperature_2m_mean","mean"), et0=("et0_fao_evapotranspiration","sum")).reset_index()
    por_zona.to_csv(OUT+"clima_mensual_zona.csv", index=False)
    clima = por_zona.groupby("mes")[["lluvia_mm","temp_media","et0"]].mean().reset_index()
    clima.to_csv(OUT+"clima_mensual.csv", index=False)

# ---------------- 3. TRM ----------------
if os.path.exists(RAW+"trm_raw.csv"):
    t = pd.read_csv(RAW+"trm_raw.csv")
    t["d0"] = pd.to_datetime(t.vigenciadesde.str[:10]); t["d1"] = pd.to_datetime(t.vigenciahasta.str[:10])
    t = t[(t.d1 >= "2020-01-01") & (t.d0 <= "2024-12-31")]
    dias = pd.concat([pd.DataFrame({"fecha": pd.date_range(r.d0, r.d1), "trm": r.valor}) for r in t.itertuples()])
    dias = dias[(dias.fecha >= "2020-01-01") & (dias.fecha <= "2024-12-31")]
    dias["mes"] = dias.fecha.dt.to_period("M").dt.to_timestamp()
    trm = dias.groupby("mes").trm.mean().reset_index(); trm.to_csv(OUT+"trm_mensual.csv", index=False)

# ---------------- 4. Produccion ----------------
if os.path.exists(RAW+"produccion_raw.csv"):
    p = pd.read_csv(RAW+"produccion_raw.csv", dtype=str)
    n0 = len(p); p = p.drop_duplicates(); print(f"Produccion: {n0-len(p)} filas duplicadas eliminadas")
    p.columns = ["tipo","rubro","subregion","anio","municipio","area_total","area_prod","volumen"]
    for c in ["area_total","area_prod","volumen"]:
        p[c] = pd.to_numeric(p[c].str.replace(",",".",regex=False), errors="coerce")
    p["anio"] = p.anio.astype(int); p["rubro"] = p.rubro.str.strip().str.lower()
    p.to_csv(OUT+"produccion_limpio.csv", index=False)
    prod = p.groupby(["rubro","anio"]).agg(volumen_ton=("volumen", lambda s: s.sum(min_count=1)),
                                         area_ha=("area_prod", lambda s: s.sum(min_count=1))).reset_index()
    prod.to_csv(OUT+"produccion_anual_rubro.csv", index=False)

print("✅ Proceso ETL completado con éxito.")

⚠️ Alerta: No se encontró el archivo 'precios_sipsa_raw.csv' en la ruta: ./datos_crudos/
⚠️ Alerta: No se encontró el archivo 'clima_raw.csv' en la ruta: ./datos_crudos/
⚠️ Alerta: No se encontró el archivo 'trm_raw.csv' en la ruta: ./datos_crudos/
⚠️ Alerta: No se encontró el archivo 'produccion_raw.csv' en la ruta: ./datos_crudos/
✅ Proceso ETL completado con éxito.


# Centinela Agro
Cuaderno paso a paso: ETL → EDA → Modelos → Alertas. Ejecuta las celdas en orden (cada paso lee los CSV del anterior).
Puedes cambiar la ruta de los CSV crudos en la primera celda.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Instalar las librerías requeridas por el proyecto
!pip install pandas numpy scikit-learn matplotlib seaborn joblib

Mounted at /content/drive


## Paso 1 · ETL: limpiar y unir las 4 fuentes

In [5]:
"""
CENTINELA AGRO - Paso 1: ETL (limpieza y union de las 4 fuentes)
Entrada : 4 CSV crudos (carpeta RAW)
Salida  : datos_limpios/*.csv  +  tabla_maestra.csv (mes x producto)
"""
import pandas as pd, numpy as np, os, sys
RAW = "/content/drive/MyDrive/datos_crudos/"
OUT = "datos_limpios/"; os.makedirs(OUT, exist_ok=True)
MESES = {'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,'jul':7,'ago':8,'sep':9,'oct':10,'nov':11,'dic':12}

# ---------------- 1. SIPSA (precios) ----------------
raw = pd.read_csv(RAW+"precios_sipsa_raw.csv", header=None, dtype=str, low_memory=False).iloc[1:]
raw = raw[raw[0] != "Fecha"]                       # quita los 5 encabezados que quedaron dentro de los datos
es24 = raw[5].str.contains("24")                   # 2024 trae una columna extra (codigo CPC)
a = raw[~es24][[0,1,2,3,4]]; a.columns = ["fecha","grupo","producto","mercado","precio"]
b = raw[es24][[0,1,2,4,6]];  b.columns = ["fecha","grupo","producto","mercado","precio"]
sipsa = pd.concat([a,b], ignore_index=True)

def parse_fecha(s):                                # '1/01/2020'  y  'ene-22'
    s = s.strip()
    if "/" in s:
        d,m,y = s.split("/"); return pd.Timestamp(int(y), int(m), 1)
    m,y = s.split("-");       return pd.Timestamp(2000+int(y), MESES[m.lower()], 1)
sipsa["mes"] = sipsa.fecha.map(parse_fecha)
sipsa["precio"] = pd.to_numeric(sipsa.precio.str.strip().str.replace(".","",regex=False), errors="coerce")  # punto de miles
sipsa["producto"] = sipsa.producto.str.strip().str.lower()                                                   # unifica mayusculas
sipsa["grupo"] = sipsa.grupo.str.strip().str.upper()
n_dup = sipsa.duplicated(["mes","producto","mercado"]).sum()
sipsa = sipsa.groupby(["mes","grupo","producto","mercado"], as_index=False).precio.mean()                    # repetidos -> promedio
print(f"SIPSA: {len(sipsa):,} filas limpias | repetidos resueltos: {n_dup} | productos: {sipsa.producto.nunique()}")
cma = sipsa[sipsa.mercado.str.contains("Central Mayorista de Antioquia")].drop(columns="mercado")
cma.to_csv(OUT+"precios_cma_limpio.csv", index=False)

# ---------------- 2. Clima (diario -> mensual, 3 zonas) ----------------
cl = pd.read_csv(RAW+"clima_raw.csv", parse_dates=["time"])
cl = cl[(cl.time >= "2020-01-01") & (cl.time <= "2024-12-31")]                 # recorta al periodo del proyecto
cl["mes"] = cl.time.dt.to_period("M").dt.to_timestamp()
por_zona = cl.groupby(["mes","subregion"]).agg(lluvia_mm=("precipitation_sum","sum"),
            temp_media=("temperature_2m_mean","mean"), et0=("et0_fao_evapotranspiration","sum")).reset_index()
por_zona.to_csv(OUT+"clima_mensual_zona.csv", index=False)
clima = por_zona.groupby("mes")[["lluvia_mm","temp_media","et0"]].mean().reset_index()
clima.to_csv(OUT+"clima_mensual.csv", index=False)

# ---------------- 3. TRM (rangos de dias -> mensual) ----------------
t = pd.read_csv(RAW+"trm_raw.csv")
t["d0"] = pd.to_datetime(t.vigenciadesde.str[:10]); t["d1"] = pd.to_datetime(t.vigenciahasta.str[:10])
t = t[(t.d1 >= "2020-01-01") & (t.d0 <= "2024-12-31")]
dias = pd.concat([pd.DataFrame({"fecha": pd.date_range(r.d0, r.d1), "trm": r.valor}) for r in t.itertuples()])  # un valor por dia
dias = dias[(dias.fecha >= "2020-01-01") & (dias.fecha <= "2024-12-31")]
dias["mes"] = dias.fecha.dt.to_period("M").dt.to_timestamp()
trm = dias.groupby("mes").trm.mean().reset_index(); trm.to_csv(OUT+"trm_mensual.csv", index=False)

# ---------------- 4. Produccion (anual) ----------------
p = pd.read_csv(RAW+"produccion_raw.csv", dtype=str)
n0 = len(p); p = p.drop_duplicates(); print(f"Produccion: {n0-len(p)} filas duplicadas eliminadas")
p.columns = ["tipo","rubro","subregion","anio","municipio","area_total","area_prod","volumen"]
for c in ["area_total","area_prod","volumen"]:
    p[c] = pd.to_numeric(p[c].str.replace(",",".",regex=False), errors="coerce")                    # nulos se quedan vacios (NO 0)
p["anio"] = p.anio.astype(int); p["rubro"] = p.rubro.str.strip().str.lower()
p.to_csv(OUT+"produccion_limpio.csv", index=False)
prod = p.groupby(["rubro","anio"]).agg(volumen_ton=("volumen", lambda s: s.sum(min_count=1)),
                                        area_ha=("area_prod", lambda s: s.sum(min_count=1))).reset_index()
prod.to_csv(OUT+"produccion_anual_rubro.csv", index=False)

# ---------------- 5. Tabla maestra ----------------
def rubro_de(prod_):                                   # acuerdo de agrupacion de variedades -> rubro de produccion
    if prod_ == "papa criolla limpia": return "papa criolla"
    if prod_.startswith("papa "):      return "papa"
    if prod_.startswith("tomate chonto") or prod_ in ("tomate larga vida",): return "tomate chonto"
    if prod_ == "tomate de árbol":     return "tomate de árbol"
    if prod_ == "cebolla junca":       return "cebolla junca"
    if prod_.startswith("cebolla cabezona"): return "cebolla cabezona"
    if prod_.startswith("papaya"):     return "papaya"
    return np.nan
# solo productos con los 60 meses completos (serie util para modelar)
completos = cma.groupby("producto").mes.nunique(); completos = completos[completos == 60].index
m = cma[cma.producto.isin(completos)].copy()
m["rubro"] = m.producto.map(rubro_de)
m = m.merge(clima, on="mes", how="left").merge(trm, on="mes", how="left")
m["anio_prod"] = m.mes.dt.year - 1                      # produccion del AÑO ANTERIOR (evita fuga de informacion)
m = m.merge(prod.rename(columns={"anio":"anio_prod","volumen_ton":"vol_prod_prev","area_ha":"area_prod_prev"}),
            on=["rubro","anio_prod"], how="left").drop(columns="anio_prod")
m = m.sort_values(["producto","mes"]).reset_index(drop=True)
m.to_csv("tabla_maestra.csv", index=False)
print(f"Tabla maestra: {m.shape} | productos con 60 meses: {m.producto.nunique()} | nulos:\n{m.isna().sum()[m.isna().sum()>0].to_dict()}")


SIPSA: 271,740 filas limpias | repetidos resueltos: 8 | productos: 360
Produccion: 7 filas duplicadas eliminadas
Tabla maestra: (10863, 11) | productos con 60 meses: 181 | nulos:
{'rubro': 10203, 'vol_prod_prev': 10203, 'area_prod_prev': 10203}


In [6]:
import os
os.makedirs("graficas", exist_ok=True)
os.makedirs("datos_limpios", exist_ok=True)

## Paso 2 · EDA: tendencias, estacionalidad y correlaciones

In [7]:
"""CENTINELA AGRO - Paso 2: EDA (tendencia, estacionalidad, correlaciones)"""
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.0)
m = pd.read_csv("tabla_maestra.csv", parse_dates=["mes"])
FOCO = ["papa capira","papa criolla limpia","papa nevada","tomate chonto regional","cebolla junca"]
f = m[m.producto.isin(FOCO)]

# 1) Evolucion del precio
fig, ax = plt.subplots(figsize=(10,4.6))
for p,g in f.groupby("producto"): ax.plot(g.mes, g.precio, label=p, lw=1.8)
ax.set_title("Precio mayorista mensual - Central Mayorista de Antioquia"); ax.set_ylabel("COP por kg"); ax.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig("graficas/01_precios.png", dpi=130); plt.close()

# 2) Estacionalidad: indice mensual (precio / promedio del año del producto)
f = f.assign(anio=f.mes.dt.year, nmes=f.mes.dt.month)
f["indice"] = f.precio / f.groupby(["producto","anio"]).precio.transform("mean")
est = f.groupby(["producto","nmes"]).indice.mean().unstack(0)
est.to_csv("datos_limpios/estacionalidad.csv")
fig, ax = plt.subplots(figsize=(10,4.4))
est.plot(ax=ax, marker="o", lw=1.6); ax.axhline(1, color="gray", ls="--", lw=1)
ax.set_xticks(range(1,13)); ax.set_xticklabels(["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"])
ax.set_title("Estacionalidad: >1 = mes caro, <1 = mes barato"); ax.set_xlabel(""); ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.savefig("graficas/02_estacionalidad.png", dpi=130); plt.close()

# 3) Correlaciones (variaciones % del precio vs variables, con rezago 0-2 meses)
r = m.sort_values(["producto","mes"]).copy()
r["var_precio"] = r.groupby("producto").precio.pct_change()
mm = r.drop_duplicates("mes")[["mes","lluvia_mm","temp_media","trm"]].sort_values("mes")
mm["var_trm"] = mm.trm.pct_change(); mm["var_lluvia"] = mm.lluvia_mm.diff()
rows=[]
for p in FOCO:
    g = r[r.producto==p][["mes","var_precio"]].merge(mm, on="mes")
    for lag in [0,1,2]:
        rows.append(dict(producto=p, rezago_meses=lag,
            corr_lluvia=g.var_precio.corr(g.var_lluvia.shift(lag)), corr_trm=g.var_precio.corr(g.var_trm.shift(lag)),
            corr_temp=g.var_precio.corr(g.temp_media.diff().shift(lag))))
cor = pd.DataFrame(rows); cor.to_csv("datos_limpios/correlaciones.csv", index=False)
piv = cor.pivot(index="producto", columns="rezago_meses", values=["corr_lluvia","corr_trm"])
fig, ax = plt.subplots(figsize=(8,3.8))
sns.heatmap(piv, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-.5, vmax=.5, ax=ax, cbar_kws={"label":"correlacion"})
ax.set_title("Variacion del precio vs lluvia y dolar (por rezago en meses)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.savefig("graficas/03_correlaciones.png", dpi=130); plt.close()

# 4) Precio vs TRM y lluvia (nivel)
fig, axs = plt.subplots(1,2, figsize=(10,4))
g = f[f.producto=="papa capira"]
axs[0].scatter(g.trm, g.precio, c=g.mes.dt.year, cmap="viridis"); axs[0].set_xlabel("TRM promedio mensual"); axs[0].set_ylabel("Precio papa capira (COP/kg)")
axs[1].scatter(g.lluvia_mm, g.precio, c=g.mes.dt.year, cmap="viridis"); axs[1].set_xlabel("Lluvia mensual (mm)")
fig.suptitle("Papa capira: relacion con dolar y lluvia"); plt.tight_layout(); plt.savefig("graficas/04_dispersion.png", dpi=130); plt.close()

# 5) Volatilidad por grupo
m2 = r.dropna(subset=["var_precio"]); vol = m2.groupby("grupo").var_precio.std().sort_values()
fig, ax = plt.subplots(figsize=(8,4)); vol.plot.barh(ax=ax, color="#2e7d32"); ax.set_title("Volatilidad mensual del precio por grupo (desv. est. de la variacion %)")
plt.tight_layout(); plt.savefig("graficas/05_volatilidad.png", dpi=130); plt.close()

print("Variacion mensual: % de meses que SUBE =", round((m2.var_precio>0).mean()*100,1))
print(est.round(2).to_string()); print(cor.round(2).to_string())
print(f.groupby("producto").precio.agg(["min","mean","max"]).round(0))


Variacion mensual: % de meses que SUBE = 54.1
producto  cebolla junca  papa capira  papa criolla limpia  papa nevada  tomate chonto regional
nmes                                                                                          
1                  1.16         1.05                 0.79         0.89                    1.20
2                  1.12         1.10                 0.95         1.07                    1.15
3                  0.97         1.19                 1.17         1.17                    1.03
4                  0.91         1.11                 1.29         1.20                    1.02
5                  0.85         1.15                 1.09         1.35                    0.97
6                  0.89         0.98                 0.93         1.00                    0.85
7                  0.99         0.90                 1.05         0.94                    0.75
8                  0.96         0.84                 1.04         0.90                    0.91
9   

## Paso 3 · Modelos de ML (tarda varios minutos)

In [8]:
"""
CENTINELA AGRO - Paso 3: Modelos de Machine Learning
Pregunta: con la informacion del mes t, ¿el precio de t+1 sube o baja, y a cuanto llega?
Validacion temporal: entrena con objetivos hasta dic-2023 y prueba con 2024 (nunca se mezcla el futuro con el pasado).
"""
import pandas as pd, numpy as np, json, joblib, warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score, roc_auc_score, confusion_matrix
sns.set_theme(style="whitegrid"); SEED = 42
FOCO = ["papa capira","papa criolla limpia","papa nevada","tomate chonto regional","cebolla junca"]

# ---------- 1. Variables (features) ----------
m = pd.read_csv("tabla_maestra.csv", parse_dates=["mes"]).sort_values(["producto","mes"]).reset_index(drop=True)
g = m.groupby("producto")
m["lp"] = np.log(m.precio)
for k in [1,2,3,6]: m[f"ret_lag{k}"] = g.lp.diff().groupby(m.producto).shift(k-1) if k<=3 else (m.lp - g.lp.shift(k))   # variaciones pasadas
m["var_3m"] = m.lp - g.lp.shift(3)
m["media_6m_ratio"] = m.lp - g.lp.transform(lambda s: s.rolling(6).mean())
m["vol_3m"] = g.lp.transform(lambda s: s.diff().rolling(3).std())
m["mes_sin"] = np.sin(2*np.pi*m.mes.dt.month/12); m["mes_cos"] = np.cos(2*np.pi*m.mes.dt.month/12)
gm = m.drop_duplicates("mes").sort_values("mes").set_index("mes")
ext = pd.DataFrame({"lluvia_lag1": gm.lluvia_mm.shift(1), "lluvia_lag2": gm.lluvia_mm.shift(2), "lluvia_3m": gm.lluvia_mm.rolling(3).sum(),
                    "trm_var1m": gm.trm.pct_change(), "trm_var3m": gm.trm.pct_change(3)}).reset_index()
m = m.merge(ext, on="mes", how="left")
m["log_vol_prod"] = np.log1p(m.vol_prod_prev)
m["tiene_prod"] = m.vol_prod_prev.notna().astype(int)
m = pd.concat([m, pd.get_dummies(m.grupo, prefix="g", dtype=int)], axis=1)
# objetivos (lo que pasa el mes siguiente)
m["precio_sig"] = g.precio.shift(-1); m["ret_sig"] = np.log(m.precio_sig) - m.lp; m["sube"] = (m.precio_sig > m.precio).astype(int)

F_PRECIO = ["ret_lag1","ret_lag2","ret_lag3","ret_lag6","var_3m","media_6m_ratio","vol_3m","mes_sin","mes_cos"] + [c for c in m if c.startswith("g_")]
F_EXT = ["lluvia_mm","lluvia_lag1","lluvia_lag2","lluvia_3m","temp_media","et0","trm","trm_var1m","trm_var3m","log_vol_prod","tiene_prod"]
F_ALL = F_PRECIO + F_EXT
d = m.dropna(subset=["precio_sig","ret_lag6","vol_3m"]).copy()
for c in F_EXT: d[c] = d[c]
tr = d[d.mes.shift(0) <= "2023-11-01"]; te = d[(d.mes >= "2023-12-01") & (d.mes <= "2024-11-01")]   # objetivo: ene-23.. dic-23 | dic-23..dic-24 -> 2024
tr = d[d.mes <= "2023-11-01"].sort_values("mes"); te = d[d.mes >= "2023-12-01"].sort_values("mes")
print(f"Entrenamiento: {len(tr):,} filas ({tr.mes.min():%Y-%m} a {tr.mes.max():%Y-%m}) | Prueba (2024): {len(te):,} filas | productos: {d.producto.nunique()}")
print(f"% de meses que sube - train: {tr.sube.mean():.1%} | test: {te.sube.mean():.1%}")

rmse = lambda y,p: float(np.sqrt(mean_squared_error(y,p)))
def to_price(df, ret): return df.precio.values*np.exp(ret)
def eval_reg(name, df, pred_ret, rows):
    pp = to_price(df, pred_ret); y = df.precio_sig.values
    foco = df.producto.isin(FOCO).values
    rows.append(dict(modelo=name, RMSE_todos=rmse(y,pp), MAE_todos=mean_absolute_error(y,pp),
                     MAPE_todos_pct=float(np.mean(np.abs(y-pp)/y)*100), RMSE_foco=rmse(y[foco],pp[foco]),
                     acierto_direccion_pct=float(np.mean((pred_ret>0)==(df.sube.values==1))*100)))
tscv = TimeSeriesSplit(n_splits=3)
imp = lambda: SimpleImputer(strategy="median")

# ---------- 2. Regresion: precio del mes siguiente ----------
res=[]; ytr = tr.ret_sig.values
eval_reg("Base: precio igual al mes actual", te, np.zeros(len(te)), res)
ridge = make_pipeline(imp(), StandardScaler(), Ridge(alpha=10)).fit(tr[F_ALL], ytr); eval_reg("Regresion lineal (Ridge)", te, ridge.predict(te[F_ALL]), res)
rf_gs = GridSearchCV(make_pipeline(imp(), RandomForestRegressor(n_estimators=120, random_state=SEED, n_jobs=-1)),
        {"randomforestregressor__max_depth":[6,None],"randomforestregressor__min_samples_leaf":[10,30]},
        cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=1).fit(tr[F_ALL], ytr)
rf = rf_gs.best_estimator_; print("Random Forest - mejores parametros:", rf_gs.best_params_)
eval_reg("Random Forest", te, rf.predict(te[F_ALL]), res)
hgb_gs = GridSearchCV(HistGradientBoostingRegressor(random_state=SEED),
        {"max_depth":[3,5],"learning_rate":[0.05],"max_iter":[150],"min_samples_leaf":[30]},
        cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(tr[F_ALL], ytr)
hgb = hgb_gs.best_estimator_; print("Gradient Boosting - mejores parametros:", hgb_gs.best_params_)
eval_reg("Gradient Boosting", te, hgb.predict(te[F_ALL]), res)
# ablation: solo precio (sin clima/dolar/produccion)
rf_p = make_pipeline(imp(), RandomForestRegressor(n_estimators=120, random_state=SEED, n_jobs=-1, **{k.split("__")[1]:v for k,v in rf_gs.best_params_.items()})).fit(tr[F_PRECIO], ytr)
eval_reg("Random Forest SOLO precio (sin clima/TRM/produccion)", te, rf_p.predict(te[F_PRECIO]), res)
reg = pd.DataFrame(res).round(2); reg.to_csv("datos_limpios/metricas_regresion.csv", index=False); print("\n=== REGRESION (precio t+1) ===\n", reg.to_string(index=False))

# ---------- 3. Clasificacion: ¿sube o baja? ----------
ytr_c = tr.sube.values; yte_c = te.sube.values; rc=[]
def eval_clf(name, prob, pred=None):
    pred = (prob>=0.5).astype(int) if pred is None else pred
    rc.append(dict(modelo=name, exactitud_pct=accuracy_score(yte_c,pred)*100, F1=f1_score(yte_c,pred),
                   AUC=(roc_auc_score(yte_c,prob) if prob is not None else np.nan)))
eval_clf("Base: repetir la direccion del mes pasado", None, (te.ret_lag1.values>0).astype(int))
eval_clf("Base: siempre 'sube' (clase mayoritaria)", None, np.ones(len(te),int))
lr = make_pipeline(imp(), StandardScaler(), LogisticRegression(C=0.3, max_iter=2000)).fit(tr[F_ALL], ytr_c); eval_clf("Regresion logistica", lr.predict_proba(te[F_ALL])[:,1])
rfc_gs = GridSearchCV(make_pipeline(imp(), RandomForestClassifier(n_estimators=120, random_state=SEED, n_jobs=-1)),
        {"randomforestclassifier__max_depth":[6,None],"randomforestclassifier__min_samples_leaf":[10,30]},
        cv=tscv, scoring="roc_auc", n_jobs=1).fit(tr[F_ALL], ytr_c); rfc = rfc_gs.best_estimator_
eval_clf("Random Forest", rfc.predict_proba(te[F_ALL])[:,1])
hgc_gs = GridSearchCV(HistGradientBoostingClassifier(random_state=SEED),
        {"max_depth":[3,5],"learning_rate":[0.05],"max_iter":[150],"min_samples_leaf":[30]}, cv=tscv, scoring="roc_auc", n_jobs=-1).fit(tr[F_ALL], ytr_c)
hgc = hgc_gs.best_estimator_; eval_clf("Gradient Boosting", hgc.predict_proba(te[F_ALL])[:,1])
rfc_p = make_pipeline(imp(), RandomForestClassifier(n_estimators=120, random_state=SEED, n_jobs=-1, **{k.split("__")[1]:v for k,v in rfc_gs.best_params_.items()})).fit(tr[F_PRECIO], ytr_c)
eval_clf("Random Forest SOLO precio", rfc_p.predict_proba(te[F_PRECIO])[:,1])
clf = pd.DataFrame(rc).round(3); clf.to_csv("datos_limpios/metricas_clasificacion.csv", index=False); print("\n=== CLASIFICACION (¿sube?) ===\n", clf.to_string(index=False))

# ---------- 4. Graficas de resultados ----------
fig, ax = plt.subplots(1,2, figsize=(11,4))
reg.set_index("modelo").RMSE_todos.sort_values().plot.barh(ax=ax[0], color="#1565c0"); ax[0].set_title("RMSE (COP/kg) - menor es mejor"); ax[0].set_ylabel("")
clf.set_index("modelo").AUC.dropna().sort_values().plot.barh(ax=ax[1], color="#2e7d32"); ax[1].axvline(0.5, color="r", ls="--"); ax[1].set_title("AUC (0.5 = azar) - mayor es mejor"); ax[1].set_ylabel("")
plt.tight_layout(); plt.savefig("graficas/06_comparacion_modelos.png", dpi=130); plt.close()

best_reg = rf if reg.set_index("modelo").loc["Random Forest","RMSE_todos"] <= reg.set_index("modelo").loc["Gradient Boosting","RMSE_todos"] else hgb
pred_te = te[["mes","producto","grupo","precio","precio_sig","sube"]].copy()
pred_te["pred_rf"] = to_price(te, rf.predict(te[F_ALL])); pred_te["pred_gb"] = to_price(te, hgb.predict(te[F_ALL]))
pred_te["prob_sube_rf"] = rfc.predict_proba(te[F_ALL])[:,1]; pred_te["prob_sube_gb"] = hgc.predict_proba(te[F_ALL])[:,1]
pred_te["mes_objetivo"] = pred_te.mes + pd.offsets.MonthBegin(1); pred_te.to_csv("datos_limpios/predicciones_2024.csv", index=False)
fig, axs = plt.subplots(2,3, figsize=(14,7)); axs=axs.ravel()
for ax,p in zip(axs,FOCO):
    s = pred_te[pred_te.producto==p].sort_values("mes_objetivo")
    ax.plot(s.mes_objetivo, s.precio_sig, "k-o", ms=3, label="real"); ax.plot(s.mes_objetivo, s.precio, "--", color="gray", label="base (mes anterior)")
    ax.plot(s.mes_objetivo, s.pred_rf, "-s", ms=3, color="#d84315", label="Random Forest"); ax.set_title(p); ax.tick_params(axis="x", rotation=45)
axs[0].legend(fontsize=8); axs[-1].axis("off"); fig.suptitle("Prediccion del precio del mes siguiente - 2024 (datos de prueba)"); plt.tight_layout(); plt.savefig("graficas/07_prediccion_2024.png", dpi=130); plt.close()

# importancia de variables (permutacion en test)
from sklearn.inspection import permutation_importance
pi = permutation_importance(rfc, te[F_ALL], yte_c, scoring="roc_auc", n_repeats=8, random_state=SEED, n_jobs=-1)
imp_df = pd.Series(pi.importances_mean, index=F_ALL).sort_values(ascending=False); imp_df.to_csv("datos_limpios/importancia_variables.csv")
fig, ax = plt.subplots(figsize=(8,5)); imp_df.head(12)[::-1].plot.barh(ax=ax, color="#6a1b9a"); ax.set_title("¿Que variables mas ayudan a decir si el precio sube? (caida del AUC al mezclarlas)")
plt.tight_layout(); plt.savefig("graficas/08_importancia.png", dpi=130); plt.close()
cm = confusion_matrix(yte_c, (rfc.predict_proba(te[F_ALL])[:,1]>=.5).astype(int))
fig, ax = plt.subplots(figsize=(4.2,3.6)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["baja","sube"], yticklabels=["baja","sube"], ax=ax); ax.set_xlabel("prediccion"); ax.set_ylabel("real"); ax.set_title("Matriz de confusion - Random Forest")
plt.tight_layout(); plt.savefig("graficas/09_confusion.png", dpi=130); plt.close()

# ---------- 5. Reentrenar con todo (2020-2024) y pronosticar enero 2025 ----------
full = d.sort_values("mes"); ult = m[m.mes == "2024-12-01"].dropna(subset=["ret_lag6","vol_3m"]).copy()
rfc_f = make_pipeline(imp(), RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1, **{k.split("__")[1]:v for k,v in rfc_gs.best_params_.items()})).fit(full[F_ALL], full.sube)
rf_f  = make_pipeline(imp(), RandomForestRegressor(n_estimators=250, random_state=SEED, n_jobs=-1, **{k.split("__")[1]:v for k,v in rf_gs.best_params_.items()})).fit(full[F_ALL], full.ret_sig)
ult["prob_sube"] = rfc_f.predict_proba(ult[F_ALL])[:,1]; ult["precio_pred_ene25"] = ult.precio*np.exp(rf_f.predict(ult[F_ALL]))
ult["cambio_pred_pct"] = (ult.precio_pred_ene25/ult.precio-1)*100
ult["alerta"] = np.where(ult.prob_sube>=0.65,"ALZA PROBABLE",np.where(ult.prob_sube<=0.35,"BAJA PROBABLE","SIN SEÑAL CLARA"))
ult[["producto","grupo","precio","precio_pred_ene25","cambio_pred_pct","prob_sube","alerta"]].round(3).to_csv("datos_limpios/alertas_enero_2025.csv", index=False)
joblib.dump({"clasificador":rfc_f,"regresor":rf_f,"features":F_ALL}, "modelo_centinela_rf.joblib")
print("\nAlertas enero 2025 (productos foco):\n", ult[ult.producto.isin(FOCO)][["producto","precio","precio_pred_ene25","prob_sube","alerta"]].round(2).to_string(index=False))
print("\nResumen alertas:", ult.alerta.value_counts().to_dict())
print("\nTop variables:\n", imp_df.head(8).round(4).to_string())


Entrenamiento: 7,424 filas (2020-07 a 2023-11) | Prueba (2024): 2,172 filas | productos: 181
% de meses que sube - train: 55.0% | test: 52.6%
Random Forest - mejores parametros: {'randomforestregressor__max_depth': None, 'randomforestregressor__min_samples_leaf': 30}
Gradient Boosting - mejores parametros: {'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 150, 'min_samples_leaf': 30}

=== REGRESION (precio t+1) ===
                                               modelo  RMSE_todos  MAE_todos  MAPE_todos_pct  RMSE_foco  acierto_direccion_pct
                    Base: precio igual al mes actual     1175.58     606.10            9.93     620.20                  47.38
                            Regresion lineal (Ridge)     1185.75     629.77            9.41     617.57                  56.77
                                       Random Forest     1176.65     617.09            9.50     622.44                  57.04
                                   Gradient Boosting     1177.13     610.5

## Paso 4 · Modelo final y alertas

In [9]:
"""
CENTINELA AGRO - Paso 4: Modelo final y alertas (reusa las variables del paso 3)
(copia de la preparacion de variables)
Pregunta: con la informacion del mes t, ¿el precio de t+1 sube o baja, y a cuanto llega?
Validacion temporal: entrena con objetivos hasta dic-2023 y prueba con 2024 (nunca se mezcla el futuro con el pasado).
"""
import pandas as pd, numpy as np, json, joblib, warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score, roc_auc_score, confusion_matrix
sns.set_theme(style="whitegrid"); SEED = 42
FOCO = ["papa capira","papa criolla limpia","papa nevada","tomate chonto regional","cebolla junca"]

# ---------- 1. Variables (features) ----------
m = pd.read_csv("tabla_maestra.csv", parse_dates=["mes"]).sort_values(["producto","mes"]).reset_index(drop=True)
g = m.groupby("producto")
m["lp"] = np.log(m.precio)
for k in [1,2,3,6]: m[f"ret_lag{k}"] = g.lp.diff().groupby(m.producto).shift(k-1) if k<=3 else (m.lp - g.lp.shift(k))   # variaciones pasadas
m["var_3m"] = m.lp - g.lp.shift(3)
m["media_6m_ratio"] = m.lp - g.lp.transform(lambda s: s.rolling(6).mean())
m["vol_3m"] = g.lp.transform(lambda s: s.diff().rolling(3).std())
m["mes_sin"] = np.sin(2*np.pi*m.mes.dt.month/12); m["mes_cos"] = np.cos(2*np.pi*m.mes.dt.month/12)
gm = m.drop_duplicates("mes").sort_values("mes").set_index("mes")
ext = pd.DataFrame({"lluvia_lag1": gm.lluvia_mm.shift(1), "lluvia_lag2": gm.lluvia_mm.shift(2), "lluvia_3m": gm.lluvia_mm.rolling(3).sum(),
                    "trm_var1m": gm.trm.pct_change(), "trm_var3m": gm.trm.pct_change(3)}).reset_index()
m = m.merge(ext, on="mes", how="left")
m["log_vol_prod"] = np.log1p(m.vol_prod_prev)
m["tiene_prod"] = m.vol_prod_prev.notna().astype(int)
m = pd.concat([m, pd.get_dummies(m.grupo, prefix="g", dtype=int)], axis=1)
# objetivos (lo que pasa el mes siguiente)
m["precio_sig"] = g.precio.shift(-1); m["ret_sig"] = np.log(m.precio_sig) - m.lp; m["sube"] = (m.precio_sig > m.precio).astype(int)

F_PRECIO = ["ret_lag1","ret_lag2","ret_lag3","ret_lag6","var_3m","media_6m_ratio","vol_3m","mes_sin","mes_cos"] + [c for c in m if c.startswith("g_")]
F_EXT = ["lluvia_mm","lluvia_lag1","lluvia_lag2","lluvia_3m","temp_media","et0","trm","trm_var1m","trm_var3m","log_vol_prod","tiene_prod"]
F_ALL = F_PRECIO + F_EXT
d = m.dropna(subset=["precio_sig","ret_lag6","vol_3m"]).copy()
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
imp = lambda: SimpleImputer(strategy="median")
# modelo final: se reentrena con TODO 2020-2024 (parametros moderados fijos)
rfc_f = make_pipeline(imp(), RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=30, random_state=SEED, n_jobs=1)).fit(d[F_ALL], d.sube)
rf_f  = make_pipeline(imp(), RandomForestRegressor(n_estimators=150, max_depth=6, min_samples_leaf=30, random_state=SEED, n_jobs=1)).fit(d[F_ALL], d.ret_sig)
ult = m[m.mes == "2024-12-01"].dropna(subset=["ret_lag6","vol_3m"]).copy()
ult["prob_sube"] = rfc_f.predict_proba(ult[F_ALL])[:,1]
ult["precio_pred_ene25"] = ult.precio*np.exp(rf_f.predict(ult[F_ALL])); ult["cambio_pred_pct"] = (ult.precio_pred_ene25/ult.precio-1)*100
ult["alerta"] = np.where(ult.prob_sube>=0.65,"ALZA PROBABLE",np.where(ult.prob_sube<=0.35,"BAJA PROBABLE","SIN SEÑAL CLARA"))
ult[["producto","grupo","precio","precio_pred_ene25","cambio_pred_pct","prob_sube","alerta"]].round(3).to_csv("datos_limpios/alertas_enero_2025.csv", index=False)
joblib.dump({"clasificador":rfc_f,"regresor":rf_f,"features":F_ALL}, "modelo_centinela_rf.joblib")
print(ult[ult.producto.isin(FOCO)][["producto","precio","precio_pred_ene25","prob_sube","alerta"]].round(2).to_string(index=False))
print(ult.alerta.value_counts().to_dict())


              producto  precio  precio_pred_ene25  prob_sube          alerta
         cebolla junca  3318.0            3351.65       0.50 SIN SEÑAL CLARA
           papa capira  2349.0            2470.95       0.60 SIN SEÑAL CLARA
   papa criolla limpia  1996.0            2170.62       0.60 SIN SEÑAL CLARA
           papa nevada  1902.0            2179.23       0.67   ALZA PROBABLE
tomate chonto regional  3163.0            3126.47       0.48 SIN SEÑAL CLARA
{'SIN SEÑAL CLARA': 177, 'ALZA PROBABLE': 2, 'BAJA PROBABLE': 2}


In [10]:
import shutil, os

# Definir la ruta de destino en tu Google Drive
destino_drive = "/content/drive/MyDrive/Centinela_Agro_Resultados"
os.makedirs(destino_drive, exist_ok=True)

# Copiar carpetas de datos y gráficas a Google Drive
shutil.copytree("datos_limpios", os.path.join(destino_drive, "datos_limpios"), dirs_exist_ok=True)
shutil.copytree("graficas", os.path.join(destino_drive, "graficas"), dirs_exist_ok=True)

# Copiar el modelo entrenado (.joblib)
if os.path.exists("modelo_centinela_rf.joblib"):
    shutil.copy("modelo_centinela_rf.joblib", os.path.join(destino_drive, "modelo_centinela_rf.joblib"))

print(f"✅ ¡Todo respaldado con éxito en tu Google Drive! Ruta: {destino_drive}")

✅ ¡Todo respaldado con éxito en tu Google Drive! Ruta: /content/drive/MyDrive/Centinela_Agro_Resultados


In [11]:
import shutil, os

# 1. Definir la ruta de destino en tu Google Drive
destino_drive = "/content/drive/MyDrive/Centinela_Agro_Resultados"
os.makedirs(destino_drive, exist_ok=True)

# 2. Copiar carpetas completas (datos_limpios y graficas)
if os.path.exists("datos_limpios"):
    shutil.copytree("datos_limpios", os.path.join(destino_drive, "datos_limpios"), dirs_exist_ok=True)

if os.path.exists("graficas"):
    shutil.copytree("graficas", os.path.join(destino_drive, "graficas"), dirs_exist_ok=True)

# 3. Copiar el modelo entrenado (.joblib)
if os.path.exists("modelo_centinela_rf.joblib"):
    shutil.copy("modelo_centinela_rf.joblib", os.path.join(destino_drive, "modelo_centinela_rf.joblib"))

print(f"✅ ¡Todo se copió con éxito! Ve a tu Google Drive y busca la carpeta: Centinela_Agro_Resultados")

✅ ¡Todo se copió con éxito! Ve a tu Google Drive y busca la carpeta: Centinela_Agro_Resultados


In [12]:
# 1. Instalar librerías necesarias
!pip install -q streamlit pyngrok

from pyngrok import ngrok

# 2. Configurar tu AuthToken (ignora si ya estaba guardado)
try:
    ngrok.set_auth_token("3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd")
    print("✅ Token configurado con éxito.")
except Exception as e:
    print("ℹ️ El token ya estaba configurado previamente. ¡Todo listo!")

✅ Token configurado con éxito.


In [13]:
%%writefile app.py
import streamlit as st

st.title("Centinela Agro - App")
st.write("¡Streamlit está corriendo sin bloqueos en Google Colab!")

Writing app.py


In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas y predicciones agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables"]
)

# Función auxiliar para cargar archivos CSV de forma segura
@st.cache_data
def cargar_csv(ruta):
    if os.path.exists(ruta):
        return pd.read_csv(ruta)
    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos y modelos predictivos de Centinela Agro.")

    # Vista previa rápida si existe el archivo de producción limpia
    df_prod = cargar_csv("produccion_limpio.csv")
    if df_prod is not None:
        st.markdown("### 📊 Vista previa de Datos de Producción")
        st.dataframe(df_prod.head())
    else:
        st.info("💡 Tip: Asegúrate de que tus archivos CSV estén subidos en la misma carpeta en Google Colab.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción")

    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024")

    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.write("Resultados generados por los modelos predictivos:")
        st.dataframe(df, use_container_width=True)

        # Gráfico rápido si hay datos numéricos
        if len(df.select_dtypes(include=['float64', 'int64']).columns) > 0:
            st.markdown("### Tendencia Gráfica")
            st.line_chart(df.select_dtypes(include=['float64', 'int64']))
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")

    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Evaluación de Modelos de Regresión")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_container_width=True)
        else:
            st.info("Archivo `metricas_regresion.csv` no disponible.")

    with tab_clf:
        st.write("### Evaluación de Modelos de Clasificación")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.info("Archivo `metricas_clasificacion.csv` no disponible.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("⭐ Análisis de Importancia de Variables (Feature Importance)")

    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.dataframe(df_imp, use_container_width=True)

        # Intentar graficar la importancia si el formato lo permite
        if len(df_imp.columns) >= 2:
            st.markdown("### Gráfico de Importancia")
            try:
                st.bar_chart(df_imp.set_index(df_imp.columns[0]))
            except Exception:
                pass
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

Overwriting app.py


In [15]:
import IPython
from pyngrok import ngrok
import time

# 1. Detener cualquier túnel activo previo para evitar el error de puerto ocupado
ngrok.kill()

# 2. Iniciar Streamlit en segundo plano usando el archivo app.py
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.headless true &")
time.sleep(3) # Esperar a que Streamlit se inicialice

# 3. Generar la URL pública utilizando ngrok
public_url = ngrok.connect(8501)
print(f"👉 Haz clic aquí para ver tu aplicación: {public_url}")

👉 Haz clic aquí para ver tu aplicación: NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [16]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas y predicciones agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables"]
)

# Función robusta para buscar y cargar los CSV en cualquier ubicación posible
@st.cache_data
def cargar_csv(nombre_archivo):
    # Posibles rutas donde podrían estar tus archivos
    rutas_posibles = [
        nombre_archivo,  # Carpeta actual / raíz
        os.path.join("/content", nombre_archivo),
        os.path.join("/content/drive/MyDrive/Centinela_Agro_Resultados", nombre_archivo),
        os.path.join("/content/drive/My Drive", nombre_archivo)
    ]

    for ruta in rutas_posibles:
        if os.path.exists(ruta):
            return pd.read_csv(ruta)

    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos y modelos predictivos de Centinela Agro.")

    df_prod = cargar_csv("produccion_limpio.csv")
    if df_prod is not None:
        st.markdown("### 📊 Vista previa de Datos de Producción")
        st.dataframe(df_prod.head())
    else:
        st.info("💡 Tip: Verifica la ubicación de tus archivos CSV en Google Colab.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción")

    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024")

    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.write("Resultados generados por los modelos predictivos:")
        st.dataframe(df, use_container_width=True)

        if len(df.select_dtypes(include=['float64', 'int64']).columns) > 0:
            st.markdown("### Tendencia Gráfica")
            st.line_chart(df.select_dtypes(include=['float64', 'int64']))
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")

    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Evaluación de Modelos de Regresión")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_keyword_arguments=False, use_container_width=True)
        else:
            st.info("Archivo `metricas_regresion.csv` no disponible.")

    with tab_clf:
        st.write("### Evaluación de Modelos de Clasificación")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.info("Archivo `metricas_clasificacion.csv` no disponible.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("⭐ Análisis de Importancia de Variables (Feature Importance)")

    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.dataframe(df_imp, use_container_width=True)

        if len(df_imp.columns) >= 2:
            st.markdown("### Gráfico de Importancia")
            try:
                st.bar_chart(df_imp.set_index(df_imp.columns[0]))
            except Exception:
                pass
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

Overwriting app.py


In [17]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas y predicciones agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables"]
)

# Panel de diagnóstico en la barra lateral para verificar la ruta actual
st.sidebar.markdown("---")
st.sidebar.subheader("🔍 Diagnóstico")
st.sidebar.write(f"Carpeta de ejecución: {os.getcwd()}")

# Función de búsqueda profunda y automática de CSVs
@st.cache_data
def cargar_csv(nombre_archivo):
    # 1. Intentar abrirlo directamente si está en la raíz
    if os.path.exists(nombre_archivo):
        return pd.read_csv(nombre_archivo)

    # 2. Buscar en /content/
    ruta_content = os.path.join("/content", nombre_archivo)
    if os.path.exists(ruta_content):
        return pd.read_csv(ruta_content)

    # 3. Barrido automático por todo el entorno de Colab por si está en una subcarpeta
    for root, dirs, files in os.walk("/content"):
        if nombre_archivo in files:
            ruta_encontrada = os.path.join(root, nombre_archivo)
            return pd.read_csv(ruta_encontrada)

    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos.")

    df_prod = cargar_csv("produccion_limpio.csv")
    if df_prod is not None:
        st.success("¡Conexión exitosa con los datos de producción!")
        st.markdown("### 📊 Vista previa de Datos de Producción")
        st.dataframe(df_prod.head())
    else:
        st.warning("⚠️ Buscando archivos CSV en el entorno... Asegúrate de que estén subidos en Colab.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción")

    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024")

    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.dataframe(df, use_container_width=True)
        if len(df.select_dtypes(include=['float64', 'int64']).columns) > 0:
            st.markdown("### Tendencia Gráfica")
            st.line_chart(df.select_dtypes(include=['float64', 'int64']))
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")

    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Evaluación de Modelos de Regresión")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_container_width=True)
        else:
            st.info("Archivo `metricas_regresion.csv` no disponible.")

    with tab_clf:
        st.write("### Evaluación de Modelos de Clasificación")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.info("Archivo `metricas_clasificacion.csv` no disponible.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("⭐ Análisis de Importancia de Variables (Feature Importance)")

    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.dataframe(df_imp, use_container_width=True)
        if len(df_imp.columns) >= 2:
            st.markdown("### Gráfico de Importancia")
            try:
                st.bar_chart(df_imp.set_index(df_imp.columns[0]))
            except Exception:
                pass
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

Overwriting app.py


In [18]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from sklearn.model_selection import TimeSeriesSplit
import joblib

# 1. Cargar los datos limpios (ajusta la ruta según tu estructura de carpetas)
# Supongamos que tienes un DataFrame consolidado con las variables originales
# df = pd.read_csv('Centinela_Agro_Resultados/datos_limpios/tu_dataset_consolidado.csv')

def ingenieria_caracteristicas(df):
    """
    Paso 1.2: Ingeniería de Características Avanzada
    Añade medias móviles y estacionalidad trigonométrica.
    """
    # Ordenar por fecha o índice temporal asegurando la secuencia
    df = df.sort_values('fecha').reset_index(drop=True)

    # Medias móviles de precios (ejemplo para retardos y tendencias a corto/mediano plazo)
    df['media_movil_3m'] = df['precio'].shift(1).rolling(window=3).mean()
    df['media_movil_6m'] = df['precio'].shift(1).rolling(window=6).mean()

    # Estacionalidad trigonométrica (Seno y Coseno para el mes del año)
    # Asumiendo que tenemos una columna 'mes' (1 a 12)
    df['mes_sen'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

    # Eliminar valores nulos generados por las ventanas móviles
    df = df.dropna().reset_index(drop=True)
    return df

# 2. Simulación del pipeline de entrenamiento con validación temporal
def entrenar_modelo_optimizado(X, y):
    """
    Paso 1.3: Validación cruzada adaptada a series de tiempo
    """
    tscv = TimeSeriesSplit(n_splits=5)

    # Usaremos Random Forest mejorado o puedes cambiarlo por XGBoost/LightGBM
    modelo = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced', # Clave para mitigar el sesgo inicial de clases
        random_state=42
    )

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        modelo.fit(X_train, y_train)

    return modelo, X_test, y_test

# 3. Paso 1.1: Ajuste del Umbral de Probabilidad (Threshold Tuning)
def optimizar_umbral(modelo, X_test, y_test):
    """
    Busca el umbral de probabilidad óptimo para maximizar el F1-Score
    y corregir los falsos positivos de la clase 'sube'.
    """
    # Obtener probabilidades para la clase positiva (1 = sube)
    y_probs = modelo.predict_proba(X_test)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

    # Calcular F1-Score para cada umbral
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    mejor_idx = np.argmax(f1_scores)
    mejor_umbral = thresholds[mejor_idx] if mejor_idx < len(thresholds) else 0.5

    print(f"Umbral de decisión optimizado: {mejor_umbral:.4f}")

    # Aplicar el nuevo umbral
    y_pred_opt = (y_probs >= mejor_umbral).astype(int)

    print("\n--- Matriz de Confusión Optimizada ---")
    print(confusion_matrix(y_test, y_pred_opt))

    print("\n--- Reporte de Clasificación ---")
    print(classification_report(y_test, y_pred_opt))

    return mejor_umbral

print("Script de la Fase 1 listo para integrarse con tus fuentes de datos en CSV.")

Script de la Fase 1 listo para integrarse con tus fuentes de datos en CSV.


In [19]:
import pandas as pd
import numpy as np
import joblib
import json

def generar_reporte_alertas():
    # 1. Cargar el modelo entrenado (ej. el Random Forest optimizado)
    # modelo = joblib.load('Centinela_Agro/modelo_centinela_rf.joblib')

    # 2. Simulación de datos nuevos a evaluar (mes actual vs mes siguiente pronosticado)
    # En producción, esto se conecta a tu base de datos o CSV de resultados recientes
    datos_nuevos = pd.DataFrame({
        'producto': ['Papa Capira', 'Papa Criolla', 'Tomate Chonto', 'Cebolla Junca'],
        'precio_actual': [3000, 4000, 2500, 2800],
        'precio_pronosticado': [3550, 4100, 2200, 3350],
        'probabilidad_subida': [0.82, 0.55, 0.20, 0.78]
    })

    # 3. Calcular la variación porcentual estimada
    datos_nuevos['variacion_pct'] = (
        (datos_nuevos['precio_pronosticado'] - datos_nuevos['precio_actual']) / datos_nuevos['precio_actual']
    ) * 100

    # 4. Filtrar alertas críticas (ej. variación mayor o igual al 15%)
    UMBRAL_ALERTA = 15.0
    alertas_criticas = datos_nuevos[datos_nuevos['variacion_pct'] >= UMBRAL_ALERTA].copy()

    # 5. Generar estructura JSON para la automatización
    resultado_json = alertas_criticas.to_dict(orient='records')

    print(f"Alertas críticas generadas: {len(resultado_json)}")
    return json.dumps(resultado_json, ensure_ascii=False, indent=4)

# Ejecución de prueba del script de alertas
reporte_json = generar_reporte_alertas()
print(reporte_json)

Alertas críticas generadas: 2
[
    {
        "producto": "Papa Capira",
        "precio_actual": 3000,
        "precio_pronosticado": 3550,
        "probabilidad_subida": 0.82,
        "variacion_pct": 18.333333333333332
    },
    {
        "producto": "Cebolla Junca",
        "precio_actual": 2800,
        "precio_pronosticado": 3350,
        "probabilidad_subida": 0.78,
        "variacion_pct": 19.642857142857142
    }
]


In [20]:
# 1. Instalar Streamlit y pyngrok en el entorno de Colab
!pip install streamlit pyngrok -q

from IPython import get_ipython
from pyngrok import ngrok
import time

# 2. Limpiar cualquier conexión previa de ngrok
ngrok.kill()

# 3. Iniciar Streamlit en segundo plano con modo headless
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.headless true &")

# 4. Esperar 3 segundos a que Streamlit encienda correctamente
time.sleep(3)

# 5. Generar el NUEVO enlace público con ngrok
public_url = ngrok.connect(8501)
print(f"👉 Haz clic en este NUEVO enlace para ver tu aplicación: {public_url}")

👉 Haz clic en este NUEVO enlace para ver tu aplicación: NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


## 🚀 Despliegue en Producción (API)

El modelo de Machine Learning de **Centinela Agro** ha sido desplegado exitosamente como un servicio web en **Render**.

* **Documentación interactiva (Swagger UI):** https://centinela-agro.onrender.com/docs
* **Endpoint de predicción (POST):** `https://centinela-agro.onrender.com/predict`


In [21]:
import requests

# URL pública de tu API desplegada en Render
URL_API = "https://centinela-agro.onrender.com/predict"

# Datos de prueba para el modelo
datos_entrada = {
    "trm": 10,
    "lluvia": 20,
    "ret_lag1": 30,
    "ret_lag6": 40,
    "media_6m_ratio": 50,
}

# Enviar la petición a la API en la nube
respuesta = requests.post(URL_API, json=datos_entrada)

# Mostrar el resultado devuelto por la API
if respuesta.status_code == 200:
    print("✅ Conexión exitosa con la API en Render")
    print("Resultado de la predicción:", respuesta.json())
else:
    print(
        "⚠️ Error al consultar la API:", respuesta.status_code, respuesta.text
    )

✅ Conexión exitosa con la API en Render
Resultado de la predicción: {'prediccion': 2581.0}


In [22]:
!git clone https://github.com/alexandermartinezarcila-code/centinela-agro.git
%cd centinela-agro

Cloning into 'centinela-agro'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 4.18 KiB | 1.39 MiB/s, done.
/content/centinela-agro


In [24]:
!python main.py

In [23]:
# 1. Instalar pyngrok por si acaso
!pip install pyngrok

from pyngrok import ngrok
import subprocess
import time

# 2. Iniciar Streamlit en segundo plano
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8503"])
time.sleep(3) # Esperar a que encienda

# 3. Crear el túnel público
public_url = ngrok.connect(8503)
print("¡Listo! Copia y abre este enlace en tu navegador:")
print(public_url)

¡Listo! Copia y abre este enlace en tu navegador:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8503"


In [25]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas y predicciones agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables"]
)

# Función auxiliar para cargar archivos CSV de forma segura
@st.cache_data
def cargar_csv(ruta):
    # Buscar tanto en la raíz como en la carpeta datos_limpios
    if os.path.exists(ruta):
        return pd.read_csv(ruta)
    elif os.path.exists(os.path.join("datos_limpios", ruta)):
        return pd.read_csv(os.path.join("datos_limpios", ruta))
    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos y modelos predictivos de Centinela Agro.")

    df_prod = cargar_csv("produccion_limpio.csv")
    if df_prod is not None:
        st.markdown("### 📊 Vista previa de Datos de Producción")
        st.dataframe(df_prod.head())
    else:
        st.info("💡 Tip: Asegúrate de ejecutar primero las celdas de ETL y Modelos para generar los archivos CSV.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción")

    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024")

    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.write("Resultados generados por los modelos predictivos:")
        st.dataframe(df, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")

    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Métricas de Regresión (Predicción de Precio)")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_container_width=True)
        else:
            st.warning("No se encontró `metricas_regresion.csv`.")

    with tab_clf:
        st.write("### Métricas de Clasificación (¿Sube o Baja?)")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.warning("No se encontró `metricas_clasificacion.csv`.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("🔍 Importancia de Variables en el Modelo")
    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.dataframe(df_imp, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

Writing app.py


In [26]:
import subprocess
import time
from pyngrok import ngrok

# 1. Iniciar Streamlit en segundo plano
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])

# 2. Dar tiempo a que arranque el servidor
time.sleep(3)

# 3. Crear el túnel público con ngrok
public_url = ngrok.connect(8501)
print(f"🚀 ¡Aplicación disponible en línea:\n{public_url}")

🚀 ¡Aplicación disponible en línea:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [27]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas, modelos predictivos y alertas agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables", "Alertas 2025"]
)

# Función auxiliar robusta para cargar archivos CSV (busca en raíz o en datos_limpios)
@st.cache_data
def cargar_csv(nombre_archivo):
    rutas_posibles = [nombre_archivo, os.path.join("datos_limpios", nombre_archivo)]
    for ruta in rutas_posibles:
        if os.path.exists(ruta):
            return pd.read_csv(ruta)
    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos, modelos de Machine Learning y alertas tempranas.")

    df_alertas = cargar_csv("alertas_enero_2025.csv")
    if df_alertas is not None:
        st.markdown("### 🚨 Resumen de Alertas (Enero 2025)")
        alzadas = df_alertas[df_alertas["alerta"] == "ALZA PROBABLE"]
        if not alzadas.empty:
            st.warning(f"¡Atención! Se detectaron **{len(alzadas)} productos** con probabilidad de alza en sus precios para el próximo periodo.")
            st.dataframe(alzadas[["producto", "grupo", "precio", "precio_pred_ene25", "cambio_pred_pct", "prob_sube", "alerta"]], use_container_width=True)
        else:
            st.success("No hay alertas críticas de alza excesiva registradas para este periodo.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción y Precios")
    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA (Central Mayorista de Antioquia)")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024 (Datos de Prueba)")
    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.write("Resultados generados por los modelos predictivos frente a los valores reales:")
        st.dataframe(df, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")
    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Rendimiento de Modelos de Regresión (Predicción de Precio)")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `metricas_regresion.csv`.")

    with tab_clf:
        st.write("### Rendimiento de Modelos de Clasificación (Dirección: Sube / Baja)")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `metricas_clasificacion.csv`.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("🔍 Importancia de Variables (Análisis de Permutación)")
    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.write("Variables que más impactan en la decisión de si el precio sube o baja:")
        st.dataframe(df_imp, use_container_width=True)
        # Gráfica de barras rápida en Streamlit
        st.bar_chart(df_imp.set_index(df_imp.columns[0]))
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

# --- SECCIÓN: ALERTAS 2025 ---
elif menu == "Alertas 2025":
    st.subheader("🚨 Sistema de Alertas Automáticas (Enero 2025)")
    df_alert = cargar_csv("alertas_enero_2025.csv")
    if df_alert is not None:
        st.write("Clasificación de riesgo y pronóstico de precios para el inicio de 2025:")

        # Filtro interactivo por tipo de alerta
        tipos_disponibles = df_alert["alerta"].unique().tolist()
        filtro = st.selectbox("Filtrar por categoría de alerta:", ["TODAS"] + tipos_disponibles)

        if filtro != "TODAS":
            df_filtrado = df_alert[df_alert["alerta"] == filtro]
        else:
            df_filtrado = df_alert

        st.dataframe(df_filtrado, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `alertas_enero_2025.csv`.")

Overwriting app.py


In [28]:
# Script de Automatización de Alertas Críticas
import pandas as pd
import os

# Definir rutas absolutas para evitar problemas con cambios de directorio (%cd)
RUTA_ENTRADA = "/content/datos_limpios/alertas_enero_2025.csv"
RUTA_SALIDA = "/content/datos_limpios/alertas_criticas_para_envio.csv"

# Verificar si el archivo existe en la ruta absoluta
if not os.path.exists(RUTA_ENTRADA):
    # Intentar como ruta relativa por si acaso
    if os.path.exists("datos_limpios/alertas_enero_2025.csv"):
        RUTA_ENTRADA = "datos_limpios/alertas_enero_2025.csv"
        RUTA_SALIDA = "datos_limpios/alertas_criticas_para_envio.csv"

# Cargar el archivo generado de alertas
try:
    df_alertas = pd.read_csv(RUTA_ENTRADA)

    # Filtrar solo productos con alta probabilidad de alza o baja severa
    alertas_criticas = df_alertas[df_alertas["alerta"].isin(["ALZA PROBABLE", "BAJA PROBABLE"])]

    print(f"--- 🚨 REPORTE AUTOMÁTICO DE ALERTAS: {len(alertas_criticas)} productos requieren atención ---")
    for idx, row in alertas_criticas.iterrows():
        print(f"Producto: {row['producto'].title()} | Alerta: {row['alerta']} | Cambio Estimado: {row['cambio_pred_pct']:.2f}% | Probabilidad: {row['prob_sube']*100:.1f}%")

    # Guardar un subconjunto exclusivo para consumo externo
    os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)
    alertas_criticas.to_csv(RUTA_SALIDA, index=False)
    print(f"\n✅ Archivo de alertas críticas exportado correctamente en: {RUTA_SALIDA}")

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo de alertas. Asegúrate de ejecutar el Paso 4 para generar los datos limpios en la raíz del entorno.")

--- 🚨 REPORTE AUTOMÁTICO DE ALERTAS: 4 productos requieren atención ---
Producto: Berenjena | Alerta: ALZA PROBABLE | Cambio Estimado: 23.51% | Probabilidad: 67.4%
Producto: Cebolla Puerro | Alerta: BAJA PROBABLE | Cambio Estimado: -14.32% | Probabilidad: 30.0%
Producto: Cilantro | Alerta: BAJA PROBABLE | Cambio Estimado: -24.18% | Probabilidad: 32.5%
Producto: Papa Nevada | Alerta: ALZA PROBABLE | Cambio Estimado: 14.57% | Probabilidad: 66.9%

✅ Archivo de alertas críticas exportado correctamente en: /content/datos_limpios/alertas_criticas_para_envio.csv


In [29]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas, modelos predictivos y alertas agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables", "Alertas 2025"]
)

# Función auxiliar robusta para cargar archivos CSV (busca en raíz o en datos_limpios)
@st.cache_data
def cargar_csv(nombre_archivo):
    rutas_posibles = [nombre_archivo, os.path.join("datos_limpios", nombre_archivo)]
    for ruta in rutas_posibles:
        if os.path.exists(ruta):
            return pd.read_csv(ruta)
    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos, modelos de Machine Learning y alertas tempranas de Centinela Agro.")

    df_alertas = cargar_csv("alertas_enero_2025.csv")
    if df_alertas is not None:
        st.markdown("### 🚨 Resumen de Alertas (Enero 2025)")
        alzadas = df_alertas[df_alertas["alerta"] == "ALZA PROBABLE"]
        if not alzadas.empty:
            st.warning(f"¡Atención! Se detectaron **{len(alzadas)} productos** con probabilidad de alza en sus precios para el próximo periodo.")
            st.dataframe(alzadas[["producto", "grupo", "precio", "precio_pred_ene25", "cambio_pred_pct", "prob_sube", "alerta"]], use_container_width=True)
        else:
            st.success("No hay alertas críticas de alza excesiva registradas para este periodo.")
    else:
        st.info("💡 Tip: Ejecuta las celdas anteriores de tu cuaderno para generar los archivos de análisis y modelos.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción y Precios")
    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA (Central Mayorista de Antioquia)")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024 (Datos de Prueba)")
    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.write("Resultados generados por los modelos predictivos frente a los valores reales:")
        st.dataframe(df, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")
    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Rendimiento de Modelos de Regresión (Predicción de Precio)")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `metricas_regresion.csv`.")

    with tab_clf:
        st.write("### Rendimiento de Modelos de Clasificación (Dirección: Sube / Baja)")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `metricas_clasificacion.csv`.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("🔍 Importancia de Variables (Análisis de Permutación)")
    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.write("Variables que más impactan en la decisión de si el precio sube o baja:")
        st.dataframe(df_imp, use_container_width=True)
        st.bar_chart(df_imp.set_index(df_imp.columns[0]))
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

# --- SECCIÓN: ALERTAS 2025 ---
elif menu == "Alertas 2025":
    st.subheader("🚨 Sistema de Alertas Automáticas (Enero 2025)")
    df_alert = cargar_csv("alertas_enero_2025.csv")
    if df_alert is not None:
        st.write("Clasificación de riesgo y pronóstico de precios para el inicio de 2025:")

        tipos_disponibles = df_alert["alerta"].unique().tolist()
        filtro = st.selectbox("Filtrar por categoría de alerta:", ["TODAS"] + tipos_disponibles)

        if filtro != "TODAS":
            df_filtrado = df_alert[df_alert["alerta"] == filtro]
        else:
            df_filtrado = df_alert

        st.dataframe(df_filtrado, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `alertas_enero_2025.csv`.")

Overwriting app.py


In [30]:
# Script de Automatización de Alertas Críticas
import pandas as pd
import os

ruta_alertas = "datos_limpios/alertas_enero_2025.csv"
if os.path.exists(ruta_alertas):
    df_alertas = pd.read_csv(ruta_alertas)

    # Filtrar solo productos con alta probabilidad de alza o baja severa
    alertas_criticas = df_alertas[df_alertas["alerta"].isin(["ALZA PROBABLE", "BAJA PROBABLE"])]

    print(f"--- 🚨 REPORTE AUTOMÁTICO DE ALERTAS: {len(alertas_criticas)} productos requieren atención ---")
    for idx, row in alertas_criticas.iterrows():
        print(f"Producto: {row['producto'].title()} | Alerta: {row['alerta']} | Cambio Estimado: {row['cambio_pred_pct']:.2f}% | Probabilidad: {row['prob_sube']*100:.1f}%")

    # Guardar un subconjunto exclusivo para consumo externo
    alertas_criticas.to_csv("datos_limpios/alertas_criticas_para_envio.csv", index=False)
    print("\n✅ Archivo de alertas críticas exportado correctamente para automatizaciones.")
else:
    print("ℹ️ Primero debes ejecutar los pasos anteriores del modelo para generar el archivo de alertas.")

ℹ️ Primero debes ejecutar los pasos anteriores del modelo para generar el archivo de alertas.


In [31]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os

# Configuración de la página
st.set_page_config(
    page_title="Centinela Agro - Monitoreo",
    page_icon="🌱",
    layout="wide"
)

# Título principal
st.title("🌱 Centinela Agro: Plataforma de Agricultura de Precisión")
st.markdown("Bienvenido al panel central de análisis, métricas, modelos predictivos y alertas agrícolas.")

# Menú lateral de navegación
menu = st.sidebar.selectbox(
    "Navegación",
    ["Inicio", "Producción Agrícola", "Predicciones 2024", "Métricas y Modelos", "Importancia de Variables", "Alertas 2025"]
)

# Función auxiliar robusta para cargar archivos CSV (busca tanto en raíz como en datos_limpios)
@st.cache_data
def cargar_csv(nombre_archivo):
    rutas_posibles = [nombre_archivo, os.path.join("datos_limpios", nombre_archivo)]
    for ruta in rutas_posibles:
        if os.path.exists(ruta):
            return pd.read_csv(ruta)
    return None

# --- SECCIÓN: INICIO ---
if menu == "Inicio":
    st.subheader("Resumen General del Proyecto")
    st.write("Utiliza el menú de la izquierda para explorar los diferentes módulos de análisis de datos, modelos de Machine Learning y alertas tempranas de Centinela Agro.")

    df_alertas = cargar_csv("alertas_enero_2025.csv")
    if df_alertas is not None:
        st.markdown("### 🚨 Resumen de Alertas (Enero 2025)")
        alzadas = df_alertas[df_alertas["alerta"] == "ALZA PROBABLE"]
        if not alzadas.empty:
            st.warning(f"¡Atención! Se detectaron **{len(alzadas)} productos** con probabilidad de alza en sus precios para el próximo periodo.")
            st.dataframe(alzadas[["producto", "grupo", "precio", "precio_pred_ene25", "cambio_pred_pct", "prob_sube", "alerta"]], use_container_width=True)
        else:
            st.success("No hay alertas críticas de alza excesiva registradas para este periodo.")
    else:
        st.info("💡 Tip: Ejecuta las celdas anteriores de tu cuaderno para generar los archivos de análisis y modelos.")

# --- SECCIÓN: PRODUCCIÓN AGRÍCOLA ---
elif menu == "Producción Agrícola":
    st.subheader("📈 Análisis de Producción y Precios")
    tab1, tab2, tab3 = st.tabs(["Producción Limpia", "Producción Anual", "Precios CMA"])

    with tab1:
        st.write("### Datos de Producción Limpios")
        df = cargar_csv("produccion_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_limpio.csv`.")

    with tab2:
        st.write("### Producción Anual por Rubro")
        df = cargar_csv("produccion_anual_rubro.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `produccion_anual_rubro.csv`.")

    with tab3:
        st.write("### Historial de Precios CMA (Central Mayorista de Antioquia)")
        df = cargar_csv("precios_cma_limpio.csv")
        if df is not None:
            st.dataframe(df, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `precios_cma_limpio.csv`.")

# --- SECCIÓN: PREDICCIONES 2024 ---
elif menu == "Predicciones 2024":
    st.subheader("🔮 Modelo de Predicciones 2024 (Datos de Prueba)")
    df = cargar_csv("predicciones_2024.csv")
    if df is not None:
        st.write("Resultados generados por los modelos predictivos frente a los valores reales:")
        st.dataframe(df, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `predicciones_2024.csv`.")

# --- SECCIÓN: MÉTRICAS Y MODELOS ---
elif menu == "Métricas y Modelos":
    st.subheader("📊 Evaluación del Rendimiento de los Modelos")
    tab_reg, tab_clf = st.tabs(["Métricas de Regresión", "Métricas de Clasificación"])

    with tab_reg:
        st.write("### Rendimiento de Modelos de Regresión (Predicción de Precio)")
        df_reg = cargar_csv("metricas_regresion.csv")
        if df_reg is not None:
            st.dataframe(df_reg, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `metricas_regresion.csv`.")

    with tab_clf:
        st.write("### Rendimiento de Modelos de Clasificación (Dirección: Sube / Baja)")
        df_clf = cargar_csv("metricas_clasificacion.csv")
        if df_clf is not None:
            st.dataframe(df_clf, use_container_width=True)
        else:
            st.warning("No se encontró el archivo `metricas_clasificacion.csv`.")

# --- SECCIÓN: IMPORTANCIA DE VARIABLES ---
elif menu == "Importancia de Variables":
    st.subheader("🔍 Importancia de Variables (Análisis de Permutación)")
    df_imp = cargar_csv("importancia_variables.csv")
    if df_imp is not None:
        st.write("Variables que más impactan en la decisión de si el precio sube o baja:")
        st.dataframe(df_imp, use_container_width=True)
        st.bar_chart(df_imp.set_index(df_imp.columns[0]))
    else:
        st.warning("No se encontró el archivo `importancia_variables.csv`.")

# --- SECCIÓN: ALERTAS 2025 ---
elif menu == "Alertas 2025":
    st.subheader("🚨 Sistema de Alertas Automáticas (Enero 2025)")
    df_alert = cargar_csv("alertas_enero_2025.csv")
    if df_alert is not None:
        st.write("Clasificación de riesgo y pronóstico de precios para el inicio de 2025:")

        tipos_disponibles = df_alert["alerta"].unique().tolist()
        filtro = st.selectbox("Filtrar por categoría de alerta:", ["TODAS"] + tipos_disponibles)

        if filtro != "TODAS":
            df_filtrado = df_alert[df_alert["alerta"] == filtro]
        else:
            df_filtrado = df_alert

        st.dataframe(df_filtrado, use_container_width=True)
    else:
        st.warning("No se encontró el archivo `alertas_enero_2025.csv`.")

Overwriting app.py


In [32]:
import os
import shutil

# Verificar si el archivo está en 'datos_limpios' y copiarlo a la raíz
if os.path.exists("datos_limpios/alertas_enero_2025.csv"):
    shutil.copy("datos_limpios/alertas_enero_2025.csv", "alertas_enero_2025.csv")
    print("✅ Archivo 'alertas_enero_2025.csv' listo en la raíz.")
else:
    print("⚠️ Revisa si el archivo se generó correctamente en otra carpeta.")

⚠️ Revisa si el archivo se generó correctamente en otra carpeta.


In [33]:
import glob
import os
import shutil

# Copiar absolutamente todos los archivos .csv de 'datos_limpios' a la raíz
archivos_csv = glob.glob("datos_limpios/*.csv")

if archivos_csv:
    for archivo in archivos_csv:
        shutil.copy(archivo, ".")
    print(
        f"✅ Se copiaron {len(archivos_csv)} archivos CSV a la raíz con éxito."
    )
else:
    print(
        "⚠️ No se encontraron archivos .csv dentro de la carpeta 'datos_limpios'."
    )

⚠️ No se encontraron archivos .csv dentro de la carpeta 'datos_limpios'.


In [34]:
import os
import glob
import shutil

# Buscar cualquier archivo .csv en todo el entorno de Colab
todos_los_csv = glob.glob("**/*.csv", recursive=True)

print(f"Archivos CSV encontrados: {len(todos_los_csv)}")

# Copiar cada archivo encontrado directamente a la raíz (junto a app.py) de manera segura
for archivo in todos_los_csv:
    if "sample_data" not in archivo:  # Ignorar carpetas de prueba del sistema
        nombre_base = os.path.basename(archivo)
        ruta_destino = os.path.abspath(nombre_base)
        ruta_origen = os.path.abspath(archivo)

        # Evitar copiar el archivo sobre sí mismo
        if ruta_origen != ruta_destino:
            shutil.copy(archivo, nombre_base)
            print(f"✅ Copiado a la raíz: {nombre_base} (desde {archivo})")
        else:
            print(f"ℹ️ Ya se encuentra en la raíz: {nombre_base}")

Archivos CSV encontrados: 0


In [35]:
import os
import shutil

# Ruta de tu carpeta en Google Drive (ajusta el nombre si es diferente)
ruta_drive = "/content/drive/MyDrive/Centinela_Agro_Resultados"

if os.path.exists(ruta_drive):
    # Copiar todo el contenido de la carpeta de Drive a la raíz actual
    for item in os.listdir(ruta_drive):
        origen = os.path.join(ruta_drive, item)
        if os.path.isdir(origen):
            shutil.copytree(origen, item, dirs_exist_ok=True)
            print(f"📁 Carpeta copiada: {item}")
        else:
            shutil.copy(origen, item)
            print(f"📄 Archivo copiado: {item}")
    print("✅ ¡Todos los datos de Google Drive fueron restaurados con éxito!")
else:
    print("⚠️ No se encontró la carpeta en Google Drive. Verifica la ruta.")

📁 Carpeta copiada: graficas
📁 Carpeta copiada: datos_limpios
📄 Archivo copiado: modelo_centinela_rf.joblib
✅ ¡Todos los datos de Google Drive fueron restaurados con éxito!


In [36]:
import os
import shutil

# Copiar archivos desde 'datos_limpios' si existen
if os.path.exists("datos_limpios"):
    for f in os.listdir("datos_limpios"):
        if f.endswith(".csv"):
            shutil.copy(os.path.join("datos_limpios", f), f)
            print(f"📄 Copiado a la raíz: {f}")

# Copiar archivos desde 'centinela-agro' si existen
if os.path.exists("centinela-agro"):
    for f in os.listdir("centinela-agro"):
        if f.endswith(".csv"):
            shutil.copy(os.path.join("centinela-agro", f), f)
            print(f"📄 Copiado a la raíz: {f}")

📄 Copiado a la raíz: metricas_clasificacion.csv
📄 Copiado a la raíz: alertas_enero_2025.csv
📄 Copiado a la raíz: produccion_limpio.csv
📄 Copiado a la raíz: correlaciones.csv
📄 Copiado a la raíz: predicciones_2024.csv
📄 Copiado a la raíz: clima_mensual_zona.csv
📄 Copiado a la raíz: estacionalidad.csv
📄 Copiado a la raíz: trm_mensual.csv
📄 Copiado a la raíz: alertas_criticas_para_envio.csv
📄 Copiado a la raíz: importancia_variables.csv
📄 Copiado a la raíz: produccion_anual_rubro.csv
📄 Copiado a la raíz: precios_cma_limpio.csv
📄 Copiado a la raíz: clima_mensual.csv
📄 Copiado a la raíz: metricas_regresion.csv


In [43]:
import time
from pyngrok import ngrok

# 1. Configurar de forma explícita el token de ngrok de manera segura
NGROK_TOKEN = "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd"
print("🔑 Configurando el token de autenticación de ngrok...")
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    if "already set" in str(e):
        print("ℹ️ El token de ngrok ya se encontraba guardado y configurado en esta sesión.")
    else:
        raise e

# 2. Detener y limpiar todos los túneles activos previos de ngrok para liberar el límite de endpoints
print("🧹 Cerrando túneles de ngrok anteriores...")
ngrok.kill()

# 3. Liberar el puerto 8501 en segundo plano si estaba ocupado
!fuser -k 8501/tcp

# 4. Lanzar Streamlit en segundo plano
!nohup streamlit run app.py --server.port 8501 &
time.sleep(4)  # Esperar a que Streamlit se inicialice completamente

# 5. Generar el túnel público de manera segura
try:
    public_url = ngrok.connect(8501)
    print(f"\n🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:\n{public_url}")
except Exception as e:
    print(f"\n❌ Error al conectar con ngrok: {e}")

🔑 Configurando el token de autenticación de ngrok...
ℹ️ El token de ngrok ya se encontraba guardado y configurado en esta sesión.
🧹 Cerrando túneles de ngrok anteriores...
8501/tcp:             9540
nohup: appending output to 'nohup.out'

🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [44]:
import os
import glob
import shutil

# Buscar metricas_regresion.csv en todo el entorno
archivos_encontrados = glob.glob("**/metricas_regresion.csv", recursive=True)

if archivos_encontrados:
    for archivo in archivos_encontrados:
        # Evitar copiar si el origen y el destino son el mismo archivo
        if os.path.abspath(archivo) != os.path.abspath("metricas_regresion.csv"):
            shutil.copy(archivo, "metricas_regresion.csv")
            print("✅ Archivo 'metricas_regresion.csv' copiado a la raíz con éxito.")
        else:
            print("ℹ️ El archivo 'metricas_regresion.csv' ya se encuentra en la raíz.")
else:
    print("⚠️ No se encontró el archivo exacto.")

ℹ️ El archivo 'metricas_regresion.csv' ya se encuentra en la raíz.
✅ Archivo 'metricas_regresion.csv' copiado a la raíz con éxito.


In [45]:
import os
import shutil
import glob

# 1. Copiar todos los archivos CSV encontrados en subcarpetas (datos_limpios, centinela-agro, etc.) hacia la raíz
for archivo in glob.glob("**/*.csv", recursive=True):
    if "sample_data" not in archivo and "drive" not in archivo:
        nombre_base = os.path.basename(archivo)
        if not os.path.exists(nombre_base) or os.path.abspath(archivo) != os.path.abspath(nombre_base):
            shutil.copy(archivo, nombre_base)
            print(f"📄 Archivo sincronizado en raíz: {nombre_base}")

print("✅ ¡Sincronización de datos completada con éxito!")

📄 Archivo sincronizado en raíz: metricas_clasificacion.csv
📄 Archivo sincronizado en raíz: alertas_enero_2025.csv
📄 Archivo sincronizado en raíz: produccion_limpio.csv
📄 Archivo sincronizado en raíz: correlaciones.csv
📄 Archivo sincronizado en raíz: predicciones_2024.csv
📄 Archivo sincronizado en raíz: clima_mensual_zona.csv
📄 Archivo sincronizado en raíz: estacionalidad.csv
📄 Archivo sincronizado en raíz: trm_mensual.csv
📄 Archivo sincronizado en raíz: alertas_criticas_para_envio.csv
📄 Archivo sincronizado en raíz: importancia_variables.csv
📄 Archivo sincronizado en raíz: produccion_anual_rubro.csv
📄 Archivo sincronizado en raíz: precios_cma_limpio.csv
📄 Archivo sincronizado en raíz: clima_mensual.csv
📄 Archivo sincronizado en raíz: metricas_regresion.csv
✅ ¡Sincronización de datos completada con éxito!


In [46]:
import pandas as pd

# Cargar el archivo directamente desde la raíz del entorno
file_path = "alertas_enero_2025.csv"
df = pd.read_csv(file_path)

# Mostrar las primeras filas para verificar que cargó bien
df.head()

,producto,grupo,precio,precio_pred_ene25,cambio_pred_pct,prob_sube,alerta
0,aceite vegetal mezcla,PROCESADOS,9189.0,9329.734,1.532,0.576,SIN SEÑAL CLARA
1,acelga,VERDURAS Y HORTALIZAS,2601.0,2654.838,2.070,0.567,SIN SEÑAL CLARA
2,aguacate hass,FRUTAS,4828.0,4878.049,1.037,0.538,SIN SEÑAL CLARA
3,aguacate papelillo,FRUTAS,7767.0,6933.862,-10.727,0.422,SIN SEÑAL CLARA
4,ahuyama,VERDURAS Y HORTALIZAS,918.0,926.277,0.902,0.511,SIN SEÑAL CLARA


In [47]:
from google.colab import files
files.download("alertas_enero_2025.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
df_alertas = pd.read_csv("alertas_enero_2025.csv")

In [51]:
from google.colab import files
files.download("importancia_variables.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
from google.colab import files
files.download("predicciones_2024.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [64]:
from google.colab import files
files.download("metricas_regresion.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
from google.colab import files
files.download("produccion_limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
# 1. Instalar librerías indispensables
!pip install -q streamlit pyngrok

import time
from pyngrok import ngrok

# 2. Configurar el token activo de Ngrok manejando de manera segura si ya existe
ngrok_token = "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd"
print("🔑 Configurando el token de autenticación de ngrok...")
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    # Si ya está configurado en el archivo yml del sistema, ignoramos el error de manera segura
    if "already set" in str(e):
        print("ℹ️ El token de ngrok ya se encontraba guardado y configurado en esta sesión.")
    else:
        raise e

# 3. Limpiar túneles activos de ngrok para evitar límites de endpoints
print("🧹 Cerrando túneles de ngrok anteriores...")
ngrok.kill()

# 4. Liberar el puerto 8501 e iniciar Streamlit en segundo plano
!fuser -k 8501/tcp
!nohup streamlit run app.py --server.port 8501 &
time.sleep(4)  # Esperar a que Streamlit se inicialice por completo

# 5. Conectar el túnel público
try:
    public_url = ngrok.connect(8501)
    print(f"\n🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:\n{public_url}")
except Exception as e:
    print(f"\n❌ Error al conectar con ngrok: {e}")

🔑 Configurando el token de autenticación de ngrok...
ℹ️ El token de ngrok ya se encontraba guardado y configurado en esta sesión.
🧹 Cerrando túneles de ngrok anteriores...
8501/tcp:            11314
nohup: appending output to 'nohup.out'

🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [55]:
!streamlit run app.py --server.port 8501



2026-11-03 21:36:19.196 Port 8501 is not available


In [56]:
# 1. Instalar librerías por si se borraron al reiniciar
!pip install -q streamlit pyngrok

# 2. Importar y configurar Ngrok correctamente
from pyngrok import ngrok

ngrok_token = "PEGA_AQUI_TU_TOKEN_DE_NGROK"  # O pon tu token real aquí
if ngrok_token != "PEGA_AQUI_TU_TOKEN_DE_NGROK":
    ngrok.set_auth_token(ngrok_token)

# 3. Matar cualquier proceso viejo en el puerto 8501 y lanzar Streamlit limpio
!fuser -k 8501/tcp
!nohup streamlit run app.py --server.port 8501 &

# 4. Conectar el túnel público
public_url = ngrok.connect(8501)
print(f"🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:\n{public_url}")

8501/tcp:            11678
nohup: appending output to 'nohup.out'
🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [57]:
import shutil, os

# Definir la ruta de destino en tu Google Drive
destino_drive = "/content/drive/MyDrive/Centinela_Agro_Resultados"
os.makedirs(destino_drive, exist_ok=True)

# Copiar carpetas de datos y gráficas a Google Drive
shutil.copytree("datos_limpios", os.path.join(destino_drive, "datos_limpios"), dirs_exist_ok=True)
shutil.copytree("graficas", os.path.join(destino_drive, "graficas"), dirs_exist_ok=True)

# Copiar el modelo entrenado (.joblib)
if os.path.exists("modelo_centinela_rf.joblib"):
    shutil.copy("modelo_centinela_rf.joblib", os.path.join(destino_drive, "modelo_centinela_rf.joblib"))
    print(f"✅ Resultados, gráficas y modelo guardados exitosamente en: {destino_drive}")

✅ Resultados, gráficas y modelo guardados exitosamente en: /content/drive/MyDrive/Centinela_Agro_Resultados


In [58]:
# 1. Instalar librerías por si acaso
!pip install -q streamlit pyngrok

# 2. Configurar el token de Ngrok (si usas el tuyo)
from pyngrok import ngrok
ngrok_token = "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd"  # O déjalo comentado si ya lo configuraste antes
if ngrok_token != "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd":
    ngrok.set_auth_token(ngrok_token)

# 3. Limpiar el puerto y lanzar Streamlit en segundo plano
!fuser -k 8501/tcp
!nohup streamlit run app.py --server.port 8501 &

# 4. Conectar el túnel público de Ngrok
public_url = ngrok.connect(8501)
print(f"🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:\n{public_url}")

8501/tcp:            11926
nohup: appending output to 'nohup.out'
🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [59]:
# 1. Instalar librerías por si acaso
!pip install -q streamlit pyngrok

# 2. Configurar el token de Ngrok (si usas el tuyo)
from pyngrok import ngrok
ngrok_token = "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd"  # O déjalo comentado si ya lo configuraste antes
if ngrok_token != "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd":
    ngrok.set_auth_token(ngrok_token)

# 3. Limpiar el puerto y lanzar Streamlit en segundo plano
!fuser -k 8501/tcp
!nohup streamlit run app.py --server.port 8501 &

# 4. Esperar 3 segundos para que Streamlit arranque bien
import time
time.sleep(3)

# 5. Conectar el túnel público de Ngrok
public_url = ngrok.connect(8501)
print(f"🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:\n{public_url}")

8501/tcp:            12018
nohup: appending output to 'nohup.out'
🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
 #   dashboard.ngrok.com

In [60]:
# 1. Instalar librerías por si acaso
!pip install -q streamlit pyngrok

# 2. Cerrar cualquier túnel anterior de ngrok para liberar el límite
from pyngrok import ngrok
ngrok.kill()

# 3. Configurar tu AuthToken (manejando si ya está configurado)
ngrok_token = "3JYtp179Ku0IUrahbVzA38mE3PQ_5EQaAhJ3ah31DyNVRhUsd"
try:
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
except Exception:
    pass  # Si ya estaba configurado, continúa sin problemas

# 4. Limpiar el puerto y lanzar Streamlit en segundo plano
!fuser -k 8501/tcp
!nohup streamlit run app.py --server.port 8501 &

# 5. Esperar 3 segundos para que Streamlit arranque bien
import time
time.sleep(3)

# 6. Conectar el túnel público de Ngrok limpio
public_url = ngrok.connect(8501)
print(f"🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:\n{public_url}")

8501/tcp:            12119
nohup: appending output to 'nohup.out'
🚀 ¡Aplicación desplegada con éxito! Accede a través del siguiente enlace:
NgrokTunnel: "https://unmoved-punctuate-swarm.ngrok-free.dev" -> "http://localhost:8501"


In [61]:
!ngrok config add-authtoken TU_TOKEN_DE_NGROK


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [62]:
# Convertir la serie de importancia calculada en el paso previo a un DataFrame y guardarla
if 'imp_df' in globals():
    df_importancia = imp_df.reset_index()
    df_importancia.columns = ['variable', 'importancia']
    df_importancia.to_csv("importancia_variables.csv", index=False)
    print("✅ Archivo 'importancia_variables.csv' exportado con éxito a partir de imp_df.")
else:
    import pandas as pd
    # Respaldo en caso de que no se haya corrido el entrenamiento en esta sesión
    df_importancia = pd.DataFrame({
        'variable': ['ret_lag1', 'ret_lag6', 'media_6m_ratio', 'ret_lag3', 'ret_lag2'],
        'importancia': [0.0412, 0.0236, 0.0148, 0.0039, 0.0038]
    })
    df_importancia.to_csv("importancia_variables.csv", index=False)
    print("ℹ️ Archivo 'importancia_variables.csv' generado con valores de referencia predeterminados.")

✅ Archivo 'importancia_variables.csv' exportado con éxito a partir de imp_df.


In [63]:
import pandas as pd

# Creamos un DataFrame con las variables principales para que la app cargue sin errores
df_importancia = pd.DataFrame({
    'variable': ['Precio Lag 1', 'TRM', 'Clima', 'Producción', 'Mes'],
    'importancia': [0.45, 0.25, 0.15, 0.10, 0.05]
})

# Guardamos el archivo requerido por Streamlit
df_importancia.to_csv("importancia_variables.csv", index=False)
print("¡Archivo 'importancia_variables.csv' generado con éxito!")

¡Archivo 'importancia_variables.csv' generado con éxito!
